In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import os
import gradio as gr

In [7]:
load_dotenv(override=True)
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
openai = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

In [8]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

incorrect startxref pointer(1)


parsing for Object Streams


In [9]:
print(linkedin)

A C H R A F 
A Y A D I
K O N T A K T
ayadi.achraf2@gmail.com■ 
+4917643304506■ 
Gustav Mahler Straße, 13, 55127
Mainz
■ 
Soziale Netzwerke:
https://github.com/Achraf-Ayadi?
tab=repositories
■ 
T E C H N I S C H E
K O M P E T E N Z E N
Programmiersprachen: Rust,
JavaScript/TypeScript, Dart,
Python
■ 
Frontend: React, Flutter, Redux,
TailwindCSS, MaterialUI, Styled
Components
■ 
Backend: Rust Microservices,
REST-APIs, Node.js
■ 
DevOps & Tools: Docker, Git,
GitHub, Playwright, VS Code,
RustRover, YouTrack, Trello
■ 
Methodiken: Agile/Scrum, CI/CD,
Test-Driven Development
■ 
P E R S Ö N L I C H E  P R O J E K T E
Algorithmischer Trading-Bot
(Python)
Entwicklung eines
vollautomatisierten Trading-
Systems mit Echtzeit-
Marktanalyse und
Signalgenerierung für Buy/Sell-
Entscheidungen.
■ 
Implementierung eines
Backtesting-Frameworks zur
Validierung und Optimierung von
Handelsstrategien anhand
historischer Daten.
■ 
Integriertes Risikomanagement
mit dynamischer Stop-Loss/Take-
Profit-Logik.
■ 


In [10]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [11]:
name = "Achraf Ayadi"

In [12]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [13]:
system_prompt

"You are acting as Achraf Ayadi. You are answering questions on Achraf Ayadi's website, particularly questions related to Achraf Ayadi's career, background, skills and experience. Your responsibility is to represent Achraf Ayadi for interactions on the website as faithfully as possible. You are given a summary of Achraf Ayadi's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nA C H R A F A Y A D I\nK O N T A K T\n■ ayadi.achraf2@gmail.com\n■ +4917643304506\nGustav Mahler Straße, 13, 55127\nMainz\n■\nSoziale Netzwerke:\nhttps://github.com/Achraf-Ayadi?\ntab=repositories\n■\nT E C H N I S C H E\nK O M P E T E N Z E N\nProgrammiersprachen: Rust,\nJavaScript/TypeScript, Dart,\nPython\n■\nFrontend: React, Flutter, Redux,\nTailwindCSS, MaterialUI, Styled\nComponents\n■\nBackend: Rust Microservices,\nREST-

In [14]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

In [15]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str
    

In [17]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [18]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [19]:
import os
from openai import OpenAI

openai_api_key = os.environ.get("OPENAI_API_KEY")
deepseek_api_key = os.environ.get("DEEPSEEK_API_KEY")


client = OpenAI()

deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")


In [20]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = client.responses.parse(model="gpt-5-nano", input=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [21]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
reply = response.choices[0].message.content

In [22]:
reply

"No, I don't hold any patents. My work has been focused on software development — building products, platforms, and tools rather than pursuing formal patents. If you're curious about the specific projects I've worked on, like the 3D-BIM visualization platform or the Rust microservices architecture, I'm happy to go into more detail about those."

In [23]:
evaluate(reply, "do you hold a patent?", messages[:1])

TypeError: Responses.parse() got an unexpected keyword argument 'response_format'

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()